## Chronos without covariates

parquet data, 4 channels, all subjects, 5 separate windows(ctx=512, h=64)

In [ ]:
# Google Times FM
!pip install "timesfm[torch]" --ignore-requires-python -q

# Amazon Chronos
!pip install git+https://github.com/amazon-science/chronos-forecasting.git -q

# pmdarima for fine tuning
!pip install pmdarima -q

# progress bar
!pip install tqdm -q

print("Restart kernel before running script further")

## Config

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
import chronos

PARQUET_PATH = os.environ.get(
    "EEG_PARQUET", os.path.join(os.getcwd(), "eeg_preprocessed_4ch_raw.parquet")
)
KAGGLE_DATASET_DIR = "/kaggle/input/datasets/michalzienkowicz/pp-eeg-4ch-raw"
PARQUET_FILENAME = "eeg_preprocessed_4ch_raw.parquet"
DATA_PATH = os.path.join(KAGGLE_DATASET_DIR, PARQUET_FILENAME)
PARQUET_PATH = DATA_PATH

LIMIT_PATIENTS = None
CHANNELS = ["Fp1", "Fp2", "P3", "P4"]
CONTEXT_LENGTH = 512
HORIZON_LENGTH = 64
N_WINDOWS = 5
FS = 500 # Native sampling rate
OFFSET_SAMPLES = 3 * FS

CHRONOS_NUM_SAMPLES = 20
CHRONOS_TEMPERATURE = 1.0
CHRONOS_TOP_P = 1.0

df = pd.read_parquet(PARQUET_PATH)
df_eval = df.head(LIMIT_PATIENTS).copy() if LIMIT_PATIENTS else df.copy()
n_total = len(df)
print(
    f"Loaded parquet: {PARQUET_PATH} | rows in file: {n_total}, "
    f"evaluation on (the first) {len(df_eval)} subjects."
)

## Model

In [ ]:
CSV_OUT = "chronos_eeg_micro_eval_results.csv"
REPO_ID = "amazon/chronos-t5-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float32

print(f"Loading Chronos ({REPO_ID}), device: {device}, dtype: {dtype}...")

try:
    model = chronos.ChronosPipeline.from_pretrained(
        REPO_ID, device_map=device, torch_dtype=dtype
    )
except TypeError:
    model = chronos.ChronosPipeline.from_pretrained(
        REPO_ID, device_map=device, dtype=dtype
    )
    
print("Chronos ready.")

def _forecasts_to_numpy(forecasts):
    if torch.is_tensor(forecasts):
        return forecasts.detach().float().cpu().numpy()
    return np.asarray(forecasts)

def forecast_windows(model, y_target, target_ch_name, starts, ctx_len, hor_len):
    batch_contexts = [torch.tensor(y_target[s : s + ctx_len], dtype=torch.float32) for s in starts]
    
    forecasts = model.predict(
        batch_contexts,
        prediction_length=hor_len,
        num_samples=CHRONOS_NUM_SAMPLES,
        temperature=CHRONOS_TEMPERATURE,
        top_p=CHRONOS_TOP_P,
    )
    
    arr = _forecasts_to_numpy(forecasts)
    preds = np.quantile(arr, 0.5, axis=1)
    
    if preds.shape != (len(starts), hor_len):
        preds = np.asarray(preds).reshape(len(starts), hor_len)
        
    return preds

## Processing functions

In [ ]:
def window_starts(total_len: int, ctx: int, hor: int, n_win: int, offset: int = OFFSET_SAMPLES) -> np.ndarray:
    need = ctx + hor
    available_len = total_len - offset
    if available_len < need:
        raise ValueError(f"Signal too short: available={available_len}, required >= {need}")
    hi = total_len - need
    
    return np.linspace(offset, hi, n_win, dtype=int)

def series_to_numpy(cell) -> np.ndarray:
    if isinstance(cell, (list, np.ndarray)):
        return np.asarray(cell, dtype=np.float64)
    return np.asarray(cell.tolist() if hasattr(cell, "tolist") else cell, dtype=np.float64)

## Evaluation loop

In [ ]:
detail_rows = []
done = 0

for _, row in df_eval.iterrows():
    sid = row["subject_id"]
    grp = row["group"] if "group" in row.index and pd.notna(row["group"]) else "Unknown"
    
    for ch in CHANNELS:
        y_target = series_to_numpy(row[ch])
        
        starts = window_starts(len(y_target), CONTEXT_LENGTH, HORIZON_LENGTH, N_WINDOWS)
            
        # --- Unified forecast call ---
        preds = forecast_windows(
            model=model, 
            y_target=y_target, 
            target_ch_name=ch, 
            starts=starts, 
            ctx_len=CONTEXT_LENGTH, 
            hor_len=HORIZON_LENGTH
        )
        
        mses_win = []
        for i, s in enumerate(starts):
            tgt_true = y_target[s + CONTEXT_LENGTH : s + CONTEXT_LENGTH + HORIZON_LENGTH]
            mses_win.append(mean_squared_error(tgt_true, preds[i]))
            
        mse_mean = float(np.mean(mses_win))
        detail_rows.append(
            {
                "record_type": "per_patient_electrode",
                "subject_id": sid,
                "group": grp,
                "electrode": ch,
                "mse": mse_mean,
                "n_windows": N_WINDOWS,
            }
        )
    done += 1
    print(f"Subjects analysed: {done} / {len(df_eval)} (last: {sid}).")

df_detail = pd.DataFrame(detail_rows)

summary_rows = []
for g in sorted(df_detail["group"].dropna().unique()):
    sub = df_detail[(df_detail["record_type"] == "per_patient_electrode") & (df_detail["group"] == g)]
    if sub.empty:
        continue
    summary_rows.append(
        {
            "record_type": "group_all_electrodes",
            "subject_id": "",
            "group": g,
            "electrode": "ALL",
            "mse": float(sub["mse"].mean()),
            "n_windows": N_WINDOWS,
        }
    )
    for ch in CHANNELS:
        sub_ch = sub[sub["electrode"] == ch]
        if sub_ch.empty:
            continue
        summary_rows.append(
            {
                "record_type": "group_per_electrode",
                "subject_id": "",
                "group": g,
                "electrode": ch,
                "mse": float(sub_ch["mse"].mean()),
                "n_windows": N_WINDOWS,
            }
        )

df_out = pd.concat([df_detail, pd.DataFrame(summary_rows)], ignore_index=True)
df_out.to_csv(CSV_OUT, index=False)
print(f"Saved: {CSV_OUT}")

try:
    from IPython.display import display
    display(df_out.tail(20))
except ImportError:
    print(df_out.tail(20).to_string())